# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset schema is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed (if running in Colab or local environment)
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and establish a reference to the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display basic information about the dataset
print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")

## 2. Data Overview
Let's examine the available record sets defined by the dataset schema. Each record set, field, and column has a unique `@id` identifier. We will print available record set `@id`s and summarize their fields by `@id` as well.

In [ ]:
# List available record sets and their field @ids
print("Record Sets in this dataset:")
for record_set in metadata.record_sets:
    print(f"  RecordSet @id: {record_set['@id']}")
    print(f"    name: {record_set.get('name', '')}")
    print(f"    description: {record_set.get('description', '')}")
    print("    Fields:")
    for field in record_set.get('field', []):
        if isinstance(field, dict):
            print(f"      - {field.get('@id', '?')} (name: {field.get('name','')})")
        else:
            print(f"      - {field} (see schema for details)")
    print("")

## 3. Data Extraction
Load data from a specific record set into a Pandas DataFrame for further analysis. All lookups are by the Croissant `@id` fields.

> **Note:** Replace `<record_set_id>` in the code below with the desired record set's `@id` from the previous step.


In [ ]:
# Select record set(s) by @id to extract records from
# If you see more than one record set above, you can add them here

# Example: 'croissant:Participants' -- Please update to the actual @id(s) from overview
record_sets_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"  No records found for {record_set_id}")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering by numeric fields, normalization, and grouping by key attributes. All field accesses reference their Croissant `@id`.

In [ ]:
# Example EDA: Choose a numeric field by its @id for demonstration
# REPLACE with actual numeric @id as found above (e.g., 'cr:GAD7_score')
if dataframes:
    # We'll use the first available dataframe for demonstration
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]

    # Print available columns
    print(f"Columns in {main_record_set_id}:\n{df.columns.tolist()}")

    # Try to find common numeric columns (for demo, guess likely field names)
    possible_numeric_fields = [c for c in df.columns if any(sub in c.lower() for sub in ['gad', 'phq', 'psq', 'score', 'age'])]

    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        # EDA: filter and normalize
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records in {main_record_set_id} with {numeric_field} > {threshold}")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a demographic field if present
        group_field_candidates = [g for g in df.columns if any(sub in g.lower() for sub in ['gender','sex','village','education'])]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No obvious numeric field found for EDA step. Please inspect DataFrame columns above.")
else:
    print("No dataframes to analyze.")

## 5. Visualization
Visualize the distribution of a numeric field and its relationship with a demographic or grouping field. Update field `@id` variables as required.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If previous numeric_field/group_field identified
if dataframes:
    try:
        if 'numeric_field' in locals() and len(df) > 0:
            plt.figure(figsize=(8,4))
            sns.histplot(df[numeric_field].dropna(), kde=True)
            plt.title(f'Distribution of {numeric_field}')
            plt.xlabel(numeric_field)
            plt.show()

        # Plot by group if possible
        if 'group_field' in locals() and group_field in df.columns:
            plt.figure(figsize=(10,4))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f'{numeric_field} by {group_field}')
            plt.ylabel(numeric_field)
            plt.xlabel(group_field)
            plt.show()
    except Exception as e:
        print(f"Plotting failed: {e}")
else:
    print("No data loaded for visualization.")

## 6. Conclusion

We have demonstrated how to load and explore the Kilifi County FAIR² Mental Health Survey dataset using the Croissant standard and `mlcroissant` library. This notebook provides examples of how to identify record sets and fields by their `@id`, extract and analyze tabular data, and visualize meaningful patterns in survey responses.

- All steps referenced schema entities by their unique `@id` for reliability and reproducibility.
- The dataset contains rich demographic and mental health assessment information, suitable for advanced data analysis and statistical modelling.
- For a full scientific analysis, refer to the dataset's accompanying documentation, schema, and the Croissant metadata file.


_This notebook can be extended to feature advanced feature engineering, predictive modelling, and reporting, all while maintaining full provenance via Croissant metadata and `mlcroissant` access._